# Classical difference-spectrum chi^2 in the MOPED-compressed space

Will's 18 Sept ask: difference split 1 and split 2, apply the (linear) MOPED
compression used by the SBI pipeline, and compute the chi^2 null test on the
resulting 5-vector. Compare against the blind full-spectrum chi^2, the SBI/KL
thresholds and the Fisher / matched-filter bound.

Everything here is analytic or uses an exact (Wishart) simulator of the two
split spectra -- no healpy, no NPE. The instrument, cosmology, compressor and
perturbation conventions are copied from the production notebooks
(`75k_*_output.ipynb`) so the comparison is like-for-like.

In [1]:
import numpy as np
import camb
from scipy import stats, optimize
from joblib import Parallel, delayed

## 1. Instrument and fiducial cosmology (identical to the production notebooks)

In [2]:
LMAX = 3000
ell = np.arange(2, LMAX + 1)                  # ell = 2..3000, shape (2999,)
M = 2 * ell + 1                               # number of real modes per ell
K_FULL = ell.size                             # 2999 dof for the blind test

FID = {'H0': 67.5, 'ombh2': 0.022, 'omch2': 0.122, 'As': 2e-9, 'ns': 0.965}
PARAM_NAMES = list(FID.keys())
DERIV_FRAC = {'H0': 0.001, 'ombh2': 0.005, 'omch2': 0.005, 'As': 0.005, 'ns': 0.005}

BEAM_FWHM_ARCMIN = 5.0
NOISE_UK_ARCMIN = 20.0
N_ell = np.full(ell.size, (NOISE_UK_ARCMIN / 60 / 180 * np.pi) ** 2)   # white noise, muK^2 sr


def gauss_beam(fwhm_arcmin):
    """B_ell for a Gaussian beam; identical to hp.gauss_beam(..., pol=False)."""
    sigma = fwhm_arcmin / 60 / 180 * np.pi / np.sqrt(8 * np.log(2))
    return np.exp(-0.5 * ell * (ell + 1) * sigma ** 2)


SIGMA_B = BEAM_FWHM_ARCMIN / 60 / 180 * np.pi / np.sqrt(8 * np.log(2))
B_ell = gauss_beam(BEAM_FWHM_ARCMIN)


def camb_cl(theta):
    """Raw TT C_ell (muK^2), ell = 2..3000. Same call as get_camb_spectrum."""
    H0, ombh2, omch2, As, ns = theta
    pars = camb.set_params(H0=H0, ombh2=ombh2, omch2=omch2, mnu=0.06, omk=0,
                           tau=0.06, As=As, ns=ns, halofit_version='mead', lmax=LMAX)
    res = camb.get_results(pars)
    return res.get_cmb_power_spectra(pars, CMB_unit='muK', raw_cl=True)['total'][2:LMAX + 1, 0]


theta_fid = np.array([FID[p] for p in PARAM_NAMES])
C_fid = camb_cl(theta_fid)
S_fid = B_ell ** 2 * C_fid                    # beamed signal S_ell = B^2 C

## 2. The MOPED compression matrix W (rebuilt exactly as `getCompression`)

compressor(d) = derivs @ ((d - fiducial) / cov) with cov = 2 C~^2/(2l+1).
So t = W (d - mu_fid), W = derivs / cov  (5 x 2999). The noise is
theta-independent so it drops out of the finite differences.

In [3]:
def moped_W():
    Ctil = S_fid + N_ell
    Sigma = 2.0 / M * Ctil ** 2
    derivs = []
    for p in PARAM_NAMES:
        step = FID[p] * DERIV_FRAC[p]
        up, dn = dict(FID), dict(FID)
        up[p] += step
        dn[p] -= step
        cu = B_ell ** 2 * camb_cl([up[q] for q in PARAM_NAMES])
        cd = B_ell ** 2 * camb_cl([dn[q] for q in PARAM_NAMES])
        derivs.append((cu - cd) / (2 * step))
    derivs = np.array(derivs)
    return derivs / Sigma, derivs, Sigma


W_raw, derivs, Sigma_d = moped_W()
# Rows differ by ~1e9 in scale (A_s). T_c is invariant to any invertible 5x5
# change of basis, so normalise rows for a well-conditioned inversion.
W = W_raw / np.linalg.norm(W_raw, axis=1, keepdims=True)

## 3. The two-split model for each systematic

Per ell: C1 = <C^1>, C2 = <C^2>, X = <a^1 a^2> (cross). Systematic on split 2.
Delta = C1 - C2, sigma^2 = Var(D) = 2/M (C1^2 + C2^2 - 2 X^2),
V = null variance = 4 N (N + 2S)/M.

In [4]:
def split_model(kind, eps, S):
    """Return (C1, C2, X) for the systematic `kind` at strength eps."""
    C1 = S + N_ell
    if kind == 'noise':          # N2 = (1+nu) N   (code: nl_null * multiplier)
        return C1, S + (1 + eps) * N_ell, S
    if kind == 'beam':           # theta_FWHM -> (1+eps) theta  =>  B2 = B^{(1+eps)^2}
        S2 = S * B_ell ** (2 * ((1 + eps) ** 2 - 1))
        return C1, S2 + N_ell, np.sqrt(S * S2)
    if kind == 'gain':           # C2 = (1+delta)(S+N) (code: cl_obs *= alpha), X = sqrt(1+delta) S
        return C1, (1 + eps) * C1, np.sqrt(1 + eps) * S
    raise ValueError(kind)


def moments(kind, eps, S):
    C1, C2, X = split_model(kind, eps, S)
    Delta = C1 - C2
    var = 2.0 / M * (C1 ** 2 + C2 ** 2 - 2 * X ** 2)
    return Delta, var


def V_null(S):
    return 4 * N_ell * (N_ell + 2 * S) / M


def dDelta_deps(kind, S):
    """Analytic dDelta/deps at eps = 0 (no linearisation of Delta itself needed)."""
    if kind == 'noise':
        return -N_ell
    if kind == 'gain':
        return -(S + N_ell)
    if kind == 'beam':
        return 2 * ell * (ell + 1) * SIGMA_B ** 2 * S
    raise ValueError(kind)


KINDS = ['gain', 'beam', 'noise']
V_fid = V_null(S_fid)

## 4. Exact simulator of (C^1, C^2, C^12) -- Bartlett decomposition of a 2x2 Wishart

(a^1_m, a^2_m) ~ N(0, [[C1, X], [X, C2]]) independently for M real modes.
M * [[C^1, C^12], [C^12, C^2]] ~ Wishart_2(M, Sigma). Bartlett: L A A^T L^T with
A = [[c1, 0], [z, c2]], c1^2 ~ chi2_M, c2^2 ~ chi2_{M-1}, z ~ N(0,1).
No Gaussian approximation anywhere.

In [5]:
def draw_spectra(C1, C2, X, n, rng):
    shape = (n, ell.size)
    c1sq = rng.chisquare(M, size=shape)
    c2sq = rng.chisquare(M - 1, size=shape)
    z = rng.standard_normal(shape)
    l11 = np.sqrt(C1)
    l21 = X / l11
    l22 = np.sqrt(np.maximum(C2 - l21 ** 2, 0.0))
    c1 = np.sqrt(c1sq)
    hat1 = l11 ** 2 * c1sq / M
    hat12 = l11 * (l21 * c1sq + l22 * c1 * z) / M
    hat2 = (l21 ** 2 * c1sq + 2 * l21 * l22 * c1 * z + l22 ** 2 * (z ** 2 + c2sq)) / M
    return hat1, hat2, hat12

## 5. Thresholds: blind full-spectrum test (reproduction) and compressed test

Detection criterion (same as SBI): median of the perturbed statistic equals
the 95th percentile of the null statistic.

In [6]:
Q95_FULL = stats.chi2.ppf(0.95, K_FULL)
LAM_CRIT_FULL = optimize.brentq(lambda l: stats.ncx2.median(K_FULL, l) - Q95_FULL, 1, 500)
K_C = 5
Q95_C = stats.chi2.ppf(0.95, K_C)
LAM_CRIT_C = optimize.brentq(lambda l: stats.ncx2.median(K_C, l) - Q95_C, 0.1, 100)

# Fixed standard-normal draws -> common random numbers, so the median is a smooth
# function of eps and the root-find is stable.
_Z5 = np.random.default_rng(2024).standard_normal((1_000_000, K_C))


def compressed_moments(kind, eps, S, Wm=None):
    """Mean and covariance of t = W D under the systematic (5-vector, 5x5)."""
    Wm = W if Wm is None else Wm
    Delta, var = moments(kind, eps, S)
    return Wm @ Delta, Wm @ (var[:, None] * Wm.T)


def Tc_median_gaussian(mu, Cp, Cinv):
    """Median of T_c = t^T C^-1 t with t ~ N(mu, Cp): generalised chi^2_5."""
    L = np.linalg.cholesky(Cp)
    t = mu + _Z5 @ L.T
    return np.median(np.einsum('ni,ij,nj->n', t, Cinv, t))


def compressed_fiducial(kind):
    Ct = W @ (V_fid[:, None] * W.T)                    # null covariance of t
    Cinv = np.linalg.inv(Ct)
    F_full = np.sum(dDelta_deps(kind, S_fid) ** 2 / V_fid)
    g = W @ dDelta_deps(kind, S_fid)
    F_c = g @ Cinv @ g
    eta = F_c / F_full

    def lam_c(e):
        mu, _ = compressed_moments(kind, e, S_fid)
        return mu @ Cinv @ mu
    e_nc = optimize.brentq(lambda e: lam_c(e) - LAM_CRIT_C, 1e-8, 0.2)

    def med(e):
        mu, Cp = compressed_moments(kind, e, S_fid)
        return Tc_median_gaussian(mu, Cp, Cinv) - Q95_C
    e_ex = optimize.brentq(med, 0.3 * e_nc, 3 * e_nc, xtol=1e-10)

    mu, Cp = compressed_moments(kind, e_ex, S_fid)
    w = np.linalg.eigvals(Cinv @ Cp).real               # weights of the generalised chi^2
    lam_full_nc = optimize.brentq(
        lambda e: np.sum(moments(kind, e, S_fid)[0] ** 2 / V_fid) - LAM_CRIT_FULL, 1e-8, 0.2)
    return dict(F_full=F_full, F_c=F_c, eta=eta,
                eps_full_nc=lam_full_nc, eps_nc=e_nc, eps_exact=e_ex,
                sig_equiv_exact=e_ex * np.sqrt(F_full),
                sig_equiv_nc=e_nc * np.sqrt(F_full),
                weights=np.sort(w), excess=np.sum(w) - K_C)

## 6. Linear-Gaussian KL proxy (same t, different metric)

For a linear-Gaussian model the posterior from split i has covariance
Sigma_post = (F + P^-1)^-1 (same for both splits) and mean shift
Delta_theta = Sigma_post t^Delta (raw compressor units). Then
D_KL = 1/2 Delta_theta^T Sigma_post^-1 Delta_theta = 1/2 t^T Sigma_post t.
Work in prior-sigma units theta' = theta / s_prior: t' = s t, F' = s F s, P'^-1 = I.

In [7]:
PRIOR_SIG = np.array([3.0, 0.001, 0.01, 0.3e-9, 0.03])
W_prior_units = PRIOR_SIG[:, None] * W_raw                    # t' = diag(s) t_raw
F_theta = W_prior_units @ (Sigma_d[:, None] * W_prior_units.T)  # Fisher of one split, prior units
A_KL = 0.5 * np.linalg.inv(F_theta + np.eye(5))                # D_KL = t'^T A_KL t'


def stat_median_gaussian(mu, Cp, A):
    L = np.linalg.cholesky(Cp)
    t = mu + _Z5 @ L.T
    return np.median(np.einsum('ni,ij,nj->n', t, A, t))


def stat_quantile_gaussian(mu, Cp, A, q):
    L = np.linalg.cholesky(Cp)
    t = mu + _Z5 @ L.T
    return np.quantile(np.einsum('ni,ij,nj->n', t, A, t), q)


def kl_proxy_fiducial(kind):
    Ct = W_prior_units @ (V_fid[:, None] * W_prior_units.T)
    q95 = stat_quantile_gaussian(np.zeros(5), Ct, A_KL, 0.95)
    F_full = np.sum(dDelta_deps(kind, S_fid) ** 2 / V_fid)

    def med(e):
        mu, Cp = compressed_moments(kind, e, S_fid, Wm=W_prior_units)
        return stat_median_gaussian(mu, Cp, A_KL) - q95
    e_guess = compressed_fiducial(kind)['eps_exact']
    e = optimize.brentq(med, 0.05 * e_guess, 50 * e_guess, xtol=1e-11)
    w = np.sort(np.linalg.eigvals(A_KL @ Ct).real)
    return dict(null_q95_KL=q95, null_mean_KL=np.trace(A_KL @ Ct), eps=e,
                sig_equiv=e * np.sqrt(F_full), null_weights=w)

## 7. Like-for-like with SBI: theta drawn from the prior in every realisation

The SBI calibration loops draw theta_i ~ prior for every null and perturbed
realisation, with the compressor (and the NPE) fixed. The analogue here:
keep W and the statistic's metric fixed at the fiducial, draw theta_i, and
calibrate the null 95th percentile and the perturbed median empirically.

(a) compressed chi^2 with C_t fixed at the fiducial
(b) linear KL proxy with its metric fixed at the fiducial
(c) compressed chi^2 with C_t estimated from the data (S_hat = cross-spectrum)
(a), (b) use Gaussian t (validated above); (c) uses the exact Wishart simulator.

In [8]:
N_PRIOR = 400
_rng_prior = np.random.default_rng(11)
theta_prior = theta_fid + PRIOR_SIG * _rng_prior.standard_normal((N_PRIOR, 5))
S_prior = B_ell ** 2 * np.array(Parallel(n_jobs=-1)(delayed(camb_cl)(th) for th in theta_prior))
V_prior = V_null(S_prior)                                        # (N_PRIOR, 2999)
print('S_ell(theta) / S_fid at ell=200: 5-95%% range', np.percentile(S_prior[:, 198] / S_fid[198], [5, 95]))

N_PER = 2500
_Zp = np.random.default_rng(99).standard_normal((N_PRIOR, N_PER, 5))


def pooled_stat(kind, eps, Wm, A):
    """Statistic t^T A t pooled over theta_i ~ prior (Gaussian t per theta_i)."""
    out = np.empty((N_PRIOR, N_PER))
    for i in range(N_PRIOR):
        Delta, var = moments(kind, eps, S_prior[i])
        mu, Cp = Wm @ Delta, Wm @ (var[:, None] * Wm.T)
        t = mu + _Zp[i] @ np.linalg.cholesky(Cp).T
        out[i] = np.einsum('ni,ij,nj->n', t, A, t)
    return out.ravel()


def prior_threshold(kind, Wm, A, e_guess):
    q95 = np.quantile(pooled_stat(kind, 0.0, Wm, A), 0.95)
    f = lambda e: np.median(pooled_stat(kind, e, Wm, A)) - q95
    e = optimize.brentq(f, 0.05 * e_guess, 50 * e_guess, xtol=1e-10)
    return q95, e


# (c) data-estimated normalisation, exact simulator
N_PER_EXACT = 60


def pooled_stat_estimated(kind, eps, seed=5):
    out = []
    for i in range(N_PRIOR):
        rng = np.random.default_rng(seed * 100_000 + i)                # common random numbers
        C1, C2, X = split_model(kind, eps, S_prior[i])
        h1, h2, h12 = draw_spectra(C1, C2, X, N_PER_EXACT, rng)
        t = (h1 - h2) @ W.T                                             # (n, 5)
        V_hat = 4 * N_ell * (N_ell + 2 * h12) / M                       # S_hat from the cross-spectrum
        Ct_hat = np.einsum('il,nl,jl->nij', W, V_hat, W)                # (n, 5, 5)
        out.append(np.einsum('ni,ni->n', t, np.linalg.solve(Ct_hat, t[..., None])[..., 0]))
    return np.concatenate(out)


def prior_threshold_estimated(kind, e_guess):
    T0 = pooled_stat_estimated(kind, 0.0)
    q95 = np.quantile(T0, 0.95)
    f = lambda e: np.median(pooled_stat_estimated(kind, e)) - q95
    e = optimize.brentq(f, 0.3 * e_guess, 5 * e_guess, xtol=1e-8)
    return T0, q95, e

S_ell(theta) / S_fid at ell=200: 5-95%% range [0.7461414  1.24507858]


## 8. Run everything

### 8a. Validation: reproduce the full-spectrum numbers from the handoff
Expected: F = 5.225e8 (gain), 3.082e7 (beam), 9.003e5 (noise);
noncentral thresholds 0.0497 %, 0.2047 %, 1.1980 %.

In [9]:
for k in KINDS:
    F = np.sum(dDelta_deps(k, S_fid) ** 2 / V_fid)
    e = optimize.brentq(lambda e: np.sum(moments(k, e, S_fid)[0] ** 2 / V_fid) - LAM_CRIT_FULL, 1e-8, 0.2)
    print(f'{k:6s} F_full = {F:.4e}   noncentral full threshold = {100*e:.4f} %')
print(f'lambda_crit: full (k=2999) = {LAM_CRIT_FULL:.2f},  compressed (k=5) = {LAM_CRIT_C:.3f}')

gain   F_full = 5.2247e+08   noncentral full threshold = 0.0497 %
beam   F_full = 3.0823e+07   noncentral full threshold = 0.2048 %
noise  F_full = 9.0034e+05   noncentral full threshold = 1.1980 %
lambda_crit: full (k=2999) = 129.21,  compressed (k=5) = 6.991


### 8b. Validation: the compressed null with the exact simulator is chi^2_5

In [10]:
_rng_val = np.random.default_rng(7)
Ct_fid = W @ (V_fid[:, None] * W.T)
Cinv_fid = np.linalg.inv(Ct_fid)


def Tc_exact_samples(kind, eps, n, rng, batch=5000):
    C1, C2, X = split_model(kind, eps, S_fid)
    out = []
    for _ in range(n // batch):
        h1, h2, _ = draw_spectra(C1, C2, X, batch, rng)
        t = (h1 - h2) @ W.T
        out.append(np.einsum('ni,ij,nj->n', t, Cinv_fid, t))
    return np.concatenate(out)


T0 = Tc_exact_samples('gain', 0.0, 100_000, _rng_val)
print(f'null T_c (exact sim, n=1e5): mean {T0.mean():.3f}  var {T0.var():.3f}  q95 {np.quantile(T0, .95):.3f}')
print(f'chi^2_5:                     mean 5.000  var 10.000  q95 {Q95_C:.3f}')

null T_c (exact sim, n=1e5): mean 4.989  var 9.931  q95 11.024
chi^2_5:                     mean 5.000  var 10.000  q95 11.070


### 8c. Compressed chi^2 at the fiducial cosmology

In [11]:
fid_res = {k: compressed_fiducial(k) for k in KINDS}
for k, r in fid_res.items():
    print(f"{k:6s} F_c = {r['F_c']:.3e}  eta = F_c/F = {r['eta']:.3f}  "
          f"threshold: noncentral {100*r['eps_nc']:.4f} %, exact {100*r['eps_exact']:.4f} %  "
          f"(sigma-equiv {r['sig_equiv_exact']:.2f});  sum(w-1) = {r['excess']:+.4f}")
for k in KINDS:
    Tp = Tc_exact_samples(k, fid_res[k]['eps_exact'], 40_000, _rng_val)
    print(f'{k:6s} exact-sim median at threshold: {np.median(Tp):.3f}  (target {Q95_C:.3f})')

gain   F_c = 9.938e+07  eta = F_c/F = 0.190  threshold: noncentral 0.0265 %, exact 0.0265 %  (sigma-equiv 6.06);  sum(w-1) = +0.0013
beam   F_c = 2.936e+07  eta = F_c/F = 0.953  threshold: noncentral 0.0488 %, exact 0.0488 %  (sigma-equiv 2.71);  sum(w-1) = -0.0022
noise  F_c = 5.050e+05  eta = F_c/F = 0.561  threshold: noncentral 0.3721 %, exact 0.3715 %  (sigma-equiv 3.53);  sum(w-1) = +0.0111


gain   exact-sim median at threshold: 11.133  (target 11.070)


beam   exact-sim median at threshold: 11.144  (target 11.070)


noise  exact-sim median at threshold: 11.086  (target 11.070)


### 8d. Linear-Gaussian KL proxy at the fiducial cosmology

In [12]:
kl_res = {k: kl_proxy_fiducial(k) for k in KINDS}
print(f"predicted null D_KL: mean {kl_res['gain']['null_mean_KL']:.3f}, 95th pct {kl_res['gain']['null_q95_KL']:.3f}"
      f"   (measured SBI null 95th pct: 2.58 - 2.78)")
print('null generalised-chi^2 weights:', np.round(kl_res['gain']['null_weights'], 4))
for k, r in kl_res.items():
    print(f"{k:6s} threshold {100*r['eps']:.4f} %  (sigma-equiv {r['sig_equiv']:.2f})")

predicted null D_KL: mean 0.822, 95th pct 2.216   (measured SBI null 95th pct: 2.58 - 2.78)
null generalised-chi^2 weights: [0.0022 0.0487 0.1376 0.1874 0.4461]
gain   threshold 0.1051 %  (sigma-equiv 24.02)
beam   threshold 0.0713 %  (sigma-equiv 3.96)
noise  threshold 0.3089 %  (sigma-equiv 2.93)


### 8e. Prior-marginalised (like-for-like with the SBI calibration loops)

In [13]:
prior_res = {}
for k in KINDS:
    eg = fid_res[k]['eps_exact']
    qa, ea = prior_threshold(k, W, Cinv_fid, eg)
    qb, eb = prior_threshold(k, W_prior_units, A_KL, eg)
    T0c, qc, ec = prior_threshold_estimated(k, eg)
    prior_res[k] = dict(q95_fixed=qa, eps_fixed=ea, q95_kl=qb, eps_kl=eb,
                        q95_est=qc, mean_est=T0c.mean(), eps_est=ec)
    print(f'{k:6s} (a) fixed C_t: q95 {qa:6.2f}, threshold {100*ea:.4f} %   '
          f'(b) KL proxy: q95 {qb:.3f}, threshold {100*eb:.4f} %   '
          f'(c) estimated C_t: null mean {T0c.mean():.2f}, q95 {qc:.2f}, threshold {100*ec:.4f} %')

gain   (a) fixed C_t: q95  11.29, threshold 0.0272 %   (b) KL proxy: q95 2.241, threshold 0.1058 %   (c) estimated C_t: null mean 5.00, q95 11.10, threshold 0.0266 %


beam   (a) fixed C_t: q95  11.29, threshold 0.0501 %   (b) KL proxy: q95 2.241, threshold 0.0723 %   (c) estimated C_t: null mean 5.00, q95 11.10, threshold 0.0491 %


noise  (a) fixed C_t: q95  11.29, threshold 0.3781 %   (b) KL proxy: q95 2.241, threshold 0.3110 %   (c) estimated C_t: null mean 5.00, q95 11.10, threshold 0.3713 %


## 9. Summary: thresholds and sigma-equivalent (sqrt(lambda_full) = eps sqrt(F))

In [14]:
SBI = {'gain': 0.00155, 'beam': 0.0008, 'noise': 0.0035}
CLASSICAL_EXACT = {'gain': 0.000485, 'beam': 0.002078, 'noise': 0.01101}   # handoff, exact weighted chi^2
rows = []
for k in KINDS:
    F = fid_res[k]['F_full']
    cols = [('Fisher/MF bound', 1.96 / np.sqrt(F)),
            ('compressed MF bound', 1.96 / np.sqrt(fid_res[k]['F_c'])),
            ('compressed chi2 (fid)', fid_res[k]['eps_exact']),
            ('compressed chi2 (prior, est. C)', prior_res[k]['eps_est']),
            ('linear KL (fid)', kl_res[k]['eps']),
            ('linear KL (prior)', prior_res[k]['eps_kl']),
            ('SBI / KL', SBI[k]),
            ('blind chi2 (exact)', CLASSICAL_EXACT[k])]
    rows.append((k, F, cols))

print(f"{'':33s}" + ''.join(f'{k:>20s}' for k in KINDS))
for j in range(len(rows[0][2])):
    name = rows[0][2][j][0]
    line = f'{name:33s}'
    for k, F, cols in rows:
        e = cols[j][1]
        line += f'{100*e:10.4f}% ({e*np.sqrt(F):5.2f}σ)'
    print(line)

                                                 gain                beam               noise
Fisher/MF bound                      0.0086% ( 1.96σ)    0.0353% ( 1.96σ)    0.2066% ( 1.96σ)
compressed MF bound                  0.0197% ( 4.49σ)    0.0362% ( 2.01σ)    0.2758% ( 2.62σ)
compressed chi2 (fid)                0.0265% ( 6.06σ)    0.0488% ( 2.71σ)    0.3715% ( 3.53σ)
compressed chi2 (prior, est. C)      0.0266% ( 6.08σ)    0.0491% ( 2.72σ)    0.3713% ( 3.52σ)
linear KL (fid)                      0.1051% (24.02σ)    0.0713% ( 3.96σ)    0.3089% ( 2.93σ)
linear KL (prior)                    0.1058% (24.18σ)    0.0723% ( 4.01σ)    0.3110% ( 2.95σ)
SBI / KL                             0.1550% (35.43σ)    0.0800% ( 4.44σ)    0.3500% ( 3.32σ)
blind chi2 (exact)                   0.0485% (11.09σ)    0.2078% (11.54σ)    1.1010% (10.45σ)
